# Listas, tuplas y conjuntos · Más listas y conjuntos

Operaciones habituales sobre listas (rotar, trocear, reordenar de forma estable, buscar el segundo mayor, detectar huecos y patrones) y un repaso de los conjuntos y las tuplas que va más allá de lo básico: métodos que modifican el conjunto in situ, relaciones entre conjuntos, `frozenset`, alias frente a copias y la mutabilidad de lo que hay dentro de una tupla. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios), [PYnative](https://pynative.com/python-exercises-with-solutions/) y [machinelearningplus](https://www.machinelearningplus.com/python/101-python-exercises/).

## 1. Rotar una lista

Rotar la lista de islas canarias $k$ posiciones a la izquierda y a la derecha con slicing, admitiendo $k$ mayor que la longitud o negativo. Hacer lo mismo con `collections.deque.rotate` (que rota a la derecha con $k$ positivo) y comprobar que coinciden.

In [1]:
from collections import deque

islas = ["Lanzarote", "Fuerteventura", "Gran Canaria", "Tenerife", "La Gomera", "La Palma", "El Hierro"]


def rotar_izquierda(lista, k):
    if not lista:
        return []
    k %= len(lista)
    return lista[k:] + lista[:k]


def rotar_derecha(lista, k):
    return rotar_izquierda(lista, -k)


print("2 a la izquierda: ", rotar_izquierda(islas, 2))
print("2 a la derecha:   ", rotar_derecha(islas, 2))
print("9 a la izquierda: ", rotar_izquierda(islas, 9))

for k in range(-20, 21):
    d = deque(islas)
    d.rotate(k)
    assert list(d) == rotar_derecha(islas, k)
    assert rotar_izquierda(rotar_derecha(islas, k), k) == islas
assert rotar_izquierda([], 3) == []

2 a la izquierda:  ['Gran Canaria', 'Tenerife', 'La Gomera', 'La Palma', 'El Hierro', 'Lanzarote', 'Fuerteventura']
2 a la derecha:    ['La Palma', 'El Hierro', 'Lanzarote', 'Fuerteventura', 'Gran Canaria', 'Tenerife', 'La Gomera']
9 a la izquierda:  ['Gran Canaria', 'Tenerife', 'La Gomera', 'La Palma', 'El Hierro', 'Lanzarote', 'Fuerteventura']


## 2. Partir en trozos

Partir una lista de 17 dorsales en trozos de $N$ elementos (el último puede ser más corto) y comprobarlo con `itertools.batched`. Después, repartirla en 3 trozos casi iguales (las longitudes difieren como mucho en 1) e invertir cada uno.

In [2]:
from itertools import batched

dorsales = list(range(101, 118))


def trozos(lista, n):
    return [lista[i:i + n] for i in range(0, len(lista), n)]


def partes_iguales(lista, partes):
    cociente, resto = divmod(len(lista), partes)
    resultado, inicio = [], 0
    for i in range(partes):
        fin = inicio + cociente + (i < resto)
        resultado.append(lista[inicio:fin])
        inicio = fin
    return resultado


print("Trozos de 5:", trozos(dorsales, 5))
tres = partes_iguales(dorsales, 3)
invertidos = [t[::-1] for t in tres]
print("3 partes:   ", tres)
print("Invertidas: ", invertidos)

for n in range(1, 20):
    assert trozos(dorsales, n) == [list(t) for t in batched(dorsales, n)]
for partes in range(1, 18):
    p = partes_iguales(dorsales, partes)
    assert sum(p, []) == dorsales and max(map(len, p)) - min(map(len, p)) <= 1
assert [x for t in invertidos for x in t[::-1]] == dorsales

Trozos de 5: [[101, 102, 103, 104, 105], [106, 107, 108, 109, 110], [111, 112, 113, 114, 115], [116, 117]]
3 partes:    [[101, 102, 103, 104, 105, 106], [107, 108, 109, 110, 111, 112], [113, 114, 115, 116, 117]]
Invertidas:  [[106, 105, 104, 103, 102, 101], [112, 111, 110, 109, 108, 107], [117, 116, 115, 114, 113]]


## 3. Reordenar sin perder el orden relativo

Mover todos los ceros de una lista al final modificándola in situ con dos índices, sin cambiar el orden de los demás elementos. Después, colocar los pares delante y los impares detrás conservando el orden de cada grupo (de forma estable). Comprobar ambas cosas con `sorted`, que es estable.

In [3]:
def ceros_al_final(lista):
    escribir = 0
    for leer in range(len(lista)):
        if lista[leer] != 0:
            lista[escribir], lista[leer] = lista[leer], lista[escribir]
            escribir += 1


def pares_delante(lista):
    return [x for x in lista if x % 2 == 0] + [x for x in lista if x % 2]


ventas = [0, 12, 0, 0, 7, 3, 0, 15, 4, 0]
original = ventas.copy()
ceros_al_final(ventas)
print("Ceros al final:", ventas)
assert ventas == sorted(original, key=lambda x: x == 0)

numeros = [7, 2, 9, 4, 4, 1, 8, 3, 6, 5]
print("Pares delante: ", pares_delante(numeros))
assert pares_delante(numeros) == sorted(numeros, key=lambda x: x % 2)

Ceros al final: [12, 7, 3, 15, 4, 0, 0, 0, 0, 0]
Pares delante:  [2, 4, 4, 8, 6, 7, 9, 1, 3, 5]


In [4]:
import random

random.seed(7)
for _ in range(500):
    datos = [random.choice([0, 0, 1, 2, 3, 4, 5]) for _ in range(random.randint(0, 12))]
    copia = datos.copy()
    ceros_al_final(copia)
    assert copia == sorted(datos, key=lambda x: x == 0)
    assert pares_delante(datos) == sorted(datos, key=lambda x: x % 2)

## 4. Segundo mayor y orden de una lista

Encontrar el segundo valor más alto de una lista en una sola pasada, sin ordenarla y teniendo en cuenta los repetidos (en `[8, 8, 5]` es 5; si todos son iguales, no hay). Después, decir si una lista está ordenada de forma ascendente, descendente o no lo está.

In [5]:
from itertools import pairwise


def segundo_mayor(lista):
    primero = segundo = None
    for x in lista:
        if primero is None or x > primero:
            primero, segundo = x, primero
        elif x != primero and (segundo is None or x > segundo):
            segundo = x
    return segundo


def tipo_de_orden(lista):
    if all(a <= b for a, b in pairwise(lista)):
        return "ascendente"
    if all(a >= b for a, b in pairwise(lista)):
        return "descendente"
    return "desordenada"


temperaturas = [24, 27, 22, 27, 25, 26]
print("Segundo mayor de", temperaturas, "→", segundo_mayor(temperaturas))
print("Segundo mayor de [5, 5, 5] →", segundo_mayor([5, 5, 5]))
for lista in ([1, 3, 3, 8], [9, 4, 4, 0], [2, 7, 1], [6]):
    print(f"{str(lista):<14} {tipo_de_orden(lista)}")

random.seed(11)
for _ in range(1000):
    datos = [random.randint(0, 9) for _ in range(random.randint(0, 8))]
    distintos = sorted(set(datos))
    assert segundo_mayor(datos) == (distintos[-2] if len(distintos) >= 2 else None)
    esperado = ("ascendente" if datos == sorted(datos)
                else "descendente" if datos == sorted(datos, reverse=True) else "desordenada")
    assert tipo_de_orden(datos) == esperado

Segundo mayor de [24, 27, 22, 27, 25, 26] → 26
Segundo mayor de [5, 5, 5] → None
[1, 3, 3, 8]   ascendente
[9, 4, 4, 0]   descendente
[2, 7, 1]      desordenada
[6]            ascendente


## 5. `zip` con longitudes distintas

Emparejar una lista de 5 guaguas con 3 conductores. Ver que `zip` se detiene en la más corta, que `zip(..., strict=True)` lanza un `ValueError` (capturarlo) y que `itertools.zip_longest` rellena los huecos con un valor dado.

In [6]:
from itertools import zip_longest

guaguas = ["L1", "L2", "L12", "L30", "L91"]
conductores = ["Ana", "Yeray", "Nayra"]

print("zip:        ", list(zip(guaguas, conductores)))
try:
    list(zip(guaguas, conductores, strict=True))
except ValueError as e:
    print("strict=True:", type(e).__name__, "-", e)
emparejadas = list(zip_longest(guaguas, conductores, fillvalue="sin asignar"))
print("zip_longest:", emparejadas)

assert len(list(zip(guaguas, conductores))) == min(len(guaguas), len(conductores))
assert len(emparejadas) == max(len(guaguas), len(conductores))
assert [c for _, c in emparejadas].count("sin asignar") == 2

zip:         [('L1', 'Ana'), ('L2', 'Yeray'), ('L12', 'Nayra')]
strict=True: ValueError - zip() argument 2 is shorter than argument 1
zip_longest: [('L1', 'Ana'), ('L2', 'Yeray'), ('L12', 'Nayra'), ('L30', 'sin asignar'), ('L91', 'sin asignar')]


## 6. De cadenas a colecciones

Convertir la cadena `"3, 8, 3, 15, 8, 42"` en una lista, una tupla y un conjunto de enteros. Obtener las palabras distintas de una frase, sin distinguir mayúsculas y en orden alfabético. Crear con una comprensión una matriz de $M \times N$ cuyo elemento $(i, j)$ valga $i \cdot j$.

In [7]:
cadena = "3, 8, 3, 15, 8, 42"
como_lista = [int(x) for x in cadena.split(",")]
como_tupla = tuple(como_lista)
como_conjunto = set(como_lista)
print(como_lista, como_tupla, sorted(como_conjunto))
assert como_conjunto == {3, 8, 15, 42} and len(como_tupla) == 6

frase = "El viento alisio sopla y el mar sopla con el viento"
palabras = sorted({p.lower() for p in frase.split()})
print(palabras)
assert palabras == sorted(set(frase.lower().split())) and len(palabras) == 7

[3, 8, 3, 15, 8, 42] (3, 8, 3, 15, 8, 42) [3, 8, 15, 42]
['alisio', 'con', 'el', 'mar', 'sopla', 'viento', 'y']


In [8]:
M, N = 4, 6
matriz = [[i * j for j in range(N)] for i in range(M)]
for fila in matriz:
    print(" ".join(f"{x:>3}" for x in fila))
assert len(matriz) == M and all(len(fila) == N for fila in matriz)
assert all(matriz[i][j] == matriz[j][i] for i in range(M) for j in range(M))

  0   0   0   0   0   0
  0   1   2   3   4   5
  0   2   4   6   8  10
  0   3   6   9  12  15


## 7. Métodos que modifican un conjunto

Partiendo del conjunto de especies vistas en una ruta, aplicar los métodos que lo modifican in situ: `update` con varios iterables a la vez, `difference_update`, `intersection_update` y `symmetric_difference_update`. Comprobar cada paso con el operador equivalente (`|`, `-`, `&`, `^`) aplicado a una copia. Por último, vaciarlo con `pop` y capturar el `KeyError` al hacer `pop` de un conjunto vacío.

In [9]:
especies = {"pinzón azul", "lagarto tizón", "cuervo"}
control = set(especies)

especies.update(["guirre", "pardela"], ("cuervo", "canario"))
control = control | {"guirre", "pardela"} | {"cuervo", "canario"}
print("update:                     ", sorted(especies))
assert especies == control

especies.difference_update({"cuervo", "gaviota"})
control = control - {"cuervo", "gaviota"}
print("difference_update:          ", sorted(especies))
assert especies == control

especies.intersection_update({"guirre", "pardela", "canario", "pinzón azul", "hubara"})
control = control & {"guirre", "pardela", "canario", "pinzón azul", "hubara"}
print("intersection_update:        ", sorted(especies))
assert especies == control

especies.symmetric_difference_update({"canario", "hubara"})
control = control ^ {"canario", "hubara"}
print("symmetric_difference_update:", sorted(especies))
assert especies == control

update:                      ['canario', 'cuervo', 'guirre', 'lagarto tizón', 'pardela', 'pinzón azul']
difference_update:           ['canario', 'guirre', 'lagarto tizón', 'pardela', 'pinzón azul']
intersection_update:         ['canario', 'guirre', 'pardela', 'pinzón azul']
symmetric_difference_update: ['guirre', 'hubara', 'pardela', 'pinzón azul']


In [10]:
sacadas = 0
while especies:
    especies.pop()
    sacadas += 1
print("Elementos sacados con pop:", sacadas)
try:
    especies.pop()
except KeyError as e:
    print("pop de un conjunto vacío:", type(e).__name__, "-", e)
assert sacadas == len(control) and not especies

Elementos sacados con pop: 4
pop de un conjunto vacío: KeyError - 'pop from an empty set'


## 8. Relaciones entre conjuntos

Con los alumnos inscritos en surf ($A$), en vela ($B$) y en buceo ($C$), comprobar con `issubset`, `issuperset` e `isdisjoint` qué relaciones hay entre ellos, distinguir subconjunto (`<=`) de subconjunto propio (`<`) y obtener quién hace surf pero ni vela ni buceo: $A - (B \cup C)$.

In [11]:
A = {"Ana", "Beatriz", "Carlos", "Dácil", "Eduardo"}
B = {"Beatriz", "Dácil"}
C = {"Fayna", "Guillermo"}
D = set(B)

print("B ⊆ A:", B.issubset(A), "  A ⊇ B:", A.issuperset(B))
print("B y C disjuntos:", B.isdisjoint(C), "  A y B disjuntos:", A.isdisjoint(B))
print("D <= B:", D <= B, "  D < B:", D < B, "  B < A:", B < A)
solo_surf = A - (B | C)
print("Solo surf:", sorted(solo_surf))

assert B.issubset(A) == (B & A == B) and A.isdisjoint(C) == (not A & C)
assert D <= B and not D < B and B < A
assert solo_surf == A - B - C == A.difference(B, C) == {x for x in A if x not in B and x not in C}

B ⊆ A: True   A ⊇ B: True
B y C disjuntos: True   A y B disjuntos: False
D <= B: True   D < B: False   B < A: True
Solo surf: ['Ana', 'Carlos', 'Eduardo']


## 9. `frozenset`, alias y copias

Usar `frozenset` como clave de un diccionario de precios de combinados de ferry, de modo que el orden de las islas no importe, y comprobar que no admite `add` (capturar el `AttributeError`). Ver la diferencia entre un alias (`b = a`) y una copia (`a.copy()`) de un conjunto. Por último, sacar con una comprensión de conjuntos las vocales distintas de una frase.

In [12]:
precios = {
    frozenset({"Tenerife", "La Gomera"}): 39.5,
    frozenset({"Gran Canaria", "Tenerife"}): 52.0,
    frozenset({"Lanzarote", "Fuerteventura"}): 24.0,
}
print("La Gomera ↔ Tenerife:", precios[frozenset({"La Gomera", "Tenerife"})], "€")

ruta = frozenset({"Lanzarote", "Fuerteventura"})
try:
    ruta.add("La Graciosa")
except AttributeError as e:
    print("frozenset.add:", e)
ampliada = ruta | {"La Graciosa"}
print("La unión devuelve otro frozenset:", type(ampliada).__name__, sorted(ampliada))
assert precios[frozenset(["Fuerteventura", "Lanzarote"])] == 24.0 and len(ruta) == 2

La Gomera ↔ Tenerife: 39.5 €
frozenset.add: 'frozenset' object has no attribute 'add'
La unión devuelve otro frozenset: frozenset ['Fuerteventura', 'La Graciosa', 'Lanzarote']


In [13]:
a = {1, 2, 3}
alias = a
copia = a.copy()
a.add(4)
print("a:", a, " alias:", alias, " copia:", copia)
print("alias is a:", alias is a, "  copia is a:", copia is a)
assert alias == {1, 2, 3, 4} and copia == {1, 2, 3}

frase = "Murmullo de olas en Famara al atardecer"
vocales = {letra for letra in frase.lower() if letra in "aeiou"}
print("Vocales distintas:", sorted(vocales))
assert vocales == set(frase.lower()) & set("aeiou")

a: {1, 2, 3, 4}  alias: {1, 2, 3, 4}  copia: {1, 2, 3}
alias is a: True   copia is a: False
Vocales distintas: ['a', 'e', 'o', 'u']


## 10. Una lista dentro de una tupla

Crear una tupla que contenga una lista y modificar esa lista: la tupla sigue siendo el mismo objeto pero su contenido cambia, porque solo guarda referencias. Comprobar que una tupla así no puede ser clave de un diccionario (`TypeError`) y comparar con `sys.getsizeof` lo que ocupa una lista y una tupla con los mismos elementos.

In [14]:
import sys

viaje = ("Agaete", ["mochila", "agua"])
mismo = viaje
viaje[1].append("toalla")
print(viaje)
print("Sigue siendo la misma tupla:", viaje is mismo)
try:
    viaje[1] = ["otra lista"]
except TypeError as e:
    print("Asignar un elemento:", e)
try:
    reservas = {viaje: "confirmada"}
except TypeError as e:
    print("Como clave:", e)
assert viaje[1] == ["mochila", "agua", "toalla"]

('Agaete', ['mochila', 'agua', 'toalla'])
Sigue siendo la misma tupla: True
Asignar un elemento: 'tuple' object does not support item assignment
Como clave: unhashable type: 'list'


In [15]:
for n in (0, 5, 100):
    datos = list(range(n))
    print(f"{n:>3} elementos: lista {sys.getsizeof(datos):>4} bytes, tupla {sys.getsizeof(tuple(datos)):>4} bytes")
    assert sys.getsizeof(tuple(datos)) < sys.getsizeof(datos)

  0 elementos: lista   56 bytes, tupla   40 bytes
  5 elementos: lista  104 bytes, tupla   80 bytes
100 elementos: lista  856 bytes, tupla  840 bytes


## 11. Números que faltan

Dada una lista ordenada de números de habitación ocupadas, obtener las que faltan entre la menor y la mayor recorriendo los pares de elementos consecutivos. Comprobarlo con una diferencia de conjuntos.

In [16]:
def faltan(ordenada):
    huecos = []
    for a, b in pairwise(ordenada):
        huecos.extend(range(a + 1, b))
    return huecos


ocupadas = [101, 102, 105, 106, 107, 110, 111, 115]
print("Libres:", faltan(ocupadas))

random.seed(3)
for _ in range(300):
    datos = sorted(random.sample(range(50), random.randint(1, 20)))
    assert faltan(datos) == sorted(set(range(datos[0], datos[-1] + 1)) - set(datos))
assert faltan([4, 4, 6]) == [5]

Libres: [103, 104, 108, 109, 112, 113, 114]


## 12. Contar bumeranes

Un bumerán es una tripleta de elementos consecutivos `a, b, a` con `b ≠ a`, como `3, 7, 3`. Contar los bumeranes de una lista (pueden solaparse: `1, 5, 1, 5` tiene dos) recorriendo los índices y con `zip` sobre tres desplazamientos.

In [17]:
def bumeranes(lista):
    encontrados = []
    for i in range(len(lista) - 2):
        if lista[i] == lista[i + 2] != lista[i + 1]:
            encontrados.append(tuple(lista[i:i + 3]))
    return encontrados


def contar_con_zip(lista):
    return sum(a == c != b for a, b, c in zip(lista, lista[1:], lista[2:]))


ejemplos = [[2, 1, 2, 3, 3, 4, 2, 4], [1, 5, 1, 5], [7, 7, 7], [9, -9, 9, -9, 9, 0]]
for lista in ejemplos:
    print(f"{str(lista):<28} {len(bumeranes(lista))}  {bumeranes(lista)}")

assert [len(bumeranes(l)) for l in ejemplos] == [2, 2, 0, 3]
random.seed(5)
for _ in range(500):
    datos = [random.randint(0, 3) for _ in range(random.randint(0, 10))]
    assert len(bumeranes(datos)) == contar_con_zip(datos)

[2, 1, 2, 3, 3, 4, 2, 4]     2  [(2, 1, 2), (4, 2, 4)]
[1, 5, 1, 5]                 2  [(1, 5, 1), (5, 1, 5)]
[7, 7, 7]                    0  []
[9, -9, 9, -9, 9, 0]         3  [(9, -9, 9), (-9, 9, -9), (9, -9, 9)]
